# Emprolijado por mesón

Adapta el CSV de **un solo mesón** para pasarlo por el código de atenuación.

Cambiar `MESON` para elegir el canal: `'PI0'`, `'ETA'`, `'RHO'`, `'OMEGA'`, `'PHI'`, `'JPSI'`

In [26]:
import numpy as np
import pandas as pd

# ---- PARÁMETRO: elegir el mesón ----
MESON = 'JPSI'  # Opciones: 'PI0', 'ETA', 'RHO', 'OMEGA', 'PHI', 'JPSI'
# ------------------------------------

input_file  = f'Data/MCP_FROM_{MESON}_SURFACE.csv'
output_file = f'Data/mCP_{MESON}_differential_flux_surface.csv'
print(f'Leyendo: {input_file}')


Leyendo: Data/MCP_FROM_JPSI_SURFACE.csv


In [27]:
df1 = pd.read_csv(input_file)
print('Columnas originales:', df1.columns.tolist())
print('Shape:', df1.shape)


Columnas originales: ['dphi_dEdCth(1/GeV/s/cm2)', 'Energy (GeV)', 'cosTh', 'm(GeV)', 'eps2']
Shape: (522000, 5)


In [28]:
# Normalizar nombres de columnas (distintos notebooks los guardan distinto)
rename_map = {
    'Energy (GeV)': 'Energy(GeV)',   # Make_MCP usa espacio
    'cosTh':        'cos_theta',      # Make_MCP usa cosTh
}
df1 = df1.rename(columns=rename_map)
print('Columnas normalizadas:', df1.columns.tolist())


Columnas normalizadas: ['dphi_dEdCth(1/GeV/s/cm2)', 'Energy(GeV)', 'cos_theta', 'm(GeV)', 'eps2']


In [29]:
# Dividir por 2π para convertir a unidades por sr
df1_og     = df1.copy()
df1_mirror = df1.copy()

df1_og['dphi_dEdCth(1/GeV/s/cm2/sr)']     = df1['dphi_dEdCth(1/GeV/s/cm2)'] / (2 * np.pi)
df1_mirror['dphi_dEdCth(1/GeV/s/cm2/sr)'] = df1['dphi_dEdCth(1/GeV/s/cm2)'] / (2 * np.pi)

# Espejo de cosenos negativos (excluir cos=0 para no duplicar)
df1_mirror = df1_mirror[df1_mirror['cos_theta'] != 0].copy()
df1_mirror['cos_theta'] = -1 * df1_mirror['cos_theta']

# Eliminar columna sin /sr
df1_og     = df1_og.drop(columns=['dphi_dEdCth(1/GeV/s/cm2)'])
df1_mirror = df1_mirror.drop(columns=['dphi_dEdCth(1/GeV/s/cm2)'])

df_mergeado = pd.concat([df1_og, df1_mirror], ignore_index=True)
print('Conteo de cos_theta:')
print(df_mergeado['cos_theta'].value_counts().sort_index())


Conteo de cos_theta:
cos_theta
-1.0    87000
-0.8    87000
-0.6    87000
-0.4    87000
-0.2    87000
 0.0    87000
 0.2    87000
 0.4    87000
 0.6    87000
 0.8    87000
 1.0    87000
Name: count, dtype: int64


In [30]:
df_mergeado.to_csv(output_file, index=False)
print(f'\nGuardado: {output_file}  ({len(df_mergeado):,} filas)')



Guardado: Data/mCP_JPSI_differential_flux_surface.csv  (957,000 filas)
